# no-transformer — training notebook

Trains the four learned modules (TF-IDF vectorizer, domain classifier, intent classifier, epistemic router MLP, confidence regressor) and packages them into `models.zip`.

**This notebook is meant to run on Google Colab.** Click **Runtime → Run all**. ~3–5 minutes on the free CPU runtime, no GPU required.

## 1. Setup

Clones the repo, installs missing deps, sets random seeds. If you fork the repo, edit `REPO_URL` below.

In [ ]:
REPO_URL = "https://github.com/Lachi-Amine/no-transformer-.git"

import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if Path("/content/repo").exists():
        os.system("rm -rf /content/repo")
    rc = os.system(f"git clone --depth 1 {REPO_URL} /content/repo")
    if rc != 0:
        raise SystemExit(f"git clone failed (exit {rc}). Check REPO_URL.")
    PROJECT = Path("/content/repo")
    os.system("pip install -q rank-bm25 pyyaml")
else:
    PROJECT = Path.cwd()
    while not (PROJECT / "pipeline" / "schemas.py").exists() and PROJECT.parent != PROJECT:
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
(PROJECT / "models").mkdir(exist_ok=True)

import random
import numpy as np
import torch
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

print(f"project: {PROJECT}")
print(f"python:  {sys.version.split()[0]}")
print(f"torch:   {torch.__version__}")
print(f"numpy:   {np.__version__}")
print(f"colab:   {IN_COLAB}")

## 2. Train domain + intent classifiers

TF-IDF + logistic regression. Saves `domain_clf.pkl`, `intent_clf.pkl`, and the shared `tfidf.pkl`.

In [ ]:
import csv
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def load_xy(path):
    X, y = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            X.append(row["text"])
            y.append(row["label"])
    return X, y

def train_clf(name, csv_path, out_path):
    X, y = load_xy(csv_path)
    print(f"\n=== {name} === {len(X)} rows, {len(set(y))} classes")
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    pipe = SkPipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf",   LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    pipe.fit(Xtr, ytr)
    print(classification_report(yte, pipe.predict(Xte), zero_division=0))
    joblib.dump(pipe, out_path)
    print(f"saved {out_path}")
    return pipe

domain_clf = train_clf("Domain", PROJECT / "training/datasets/domains.csv",
                       PROJECT / "models/domain_clf.pkl")
intent_clf = train_clf("Intent", PROJECT / "training/datasets/intents.csv",
                       PROJECT / "models/intent_clf.pkl")

joblib.dump(domain_clf.named_steps["tfidf"], PROJECT / "models/tfidf.pkl")
print("saved tfidf.pkl")

## 3. Train epistemic router MLP

Small MLP (input → 64 → 32 → 3, softmax) trained with cross-entropy on the soft `g/y/r` labels in `epistemic.csv`.

In [ ]:
import csv
import json
import re
import numpy as np
import torch
import torch.optim as optim
from sklearn.model_selection import train_test_split

from pipeline.schemas import Classification, Query
from pipeline.features import router_features
from pipeline.router import build_mlp

TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_-]*")

def make_query(text):
    norm = text.strip().lower()
    return Query(raw=text, normalized=norm,
                 tokens=tuple(TOKEN_RE.findall(norm)), entities={})

def make_cls(domain, intent):
    return Classification(domain=domain, intent=intent,
                          domain_probs={}, intent_probs={})

rows = []
with open(PROJECT / "training/datasets/epistemic.csv", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append(r)
print(f"epistemic.csv: {len(rows)} rows")

X = np.stack([
    router_features(make_query(r["text"]), make_cls(r["domain"], r["intent"]))
    for r in rows
]).astype(np.float32)
Y = np.array(
    [[float(r["g"]), float(r["y"]), float(r["r"])] for r in rows],
    dtype=np.float32,
)
INPUT_DIM = X.shape[1]
print(f"X: {X.shape}, Y: {Y.shape}, input_dim: {INPUT_DIM}")

Xtr, Xte, ytr, yte = train_test_split(X, Y, test_size=0.2, random_state=0)
Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr)
Xte_t = torch.from_numpy(Xte); yte_t = torch.from_numpy(yte)

torch.manual_seed(0)
model = build_mlp(INPUT_DIM)
opt = optim.Adam(model.parameters(), lr=1e-3)

def soft_xent(pred, target):
    return -(target * (pred + 1e-9).log()).sum(dim=-1).mean()

EPOCHS = 60
BATCH = 32
n = Xtr_t.shape[0]
best_val = float("inf")
best_state = None

for epoch in range(EPOCHS):
    model.train()
    idx = torch.randperm(n)
    epoch_loss = 0.0
    for i in range(0, n, BATCH):
        b = idx[i:i + BATCH]
        loss = soft_xent(model(Xtr_t[b]), ytr_t[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * len(b)
    epoch_loss /= n
    model.eval()
    with torch.no_grad():
        val_loss = soft_xent(model(Xte_t), yte_t).item()
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train={epoch_loss:.4f}  val={val_loss:.4f}")

model.load_state_dict(best_state)
torch.save(model.state_dict(), PROJECT / "models/epistemic_router.pt")
(PROJECT / "models/epistemic_router.json").write_text(
    json.dumps({"input_dim": INPUT_DIM})
)
print(f"\nsaved epistemic_router.pt (best val loss {best_val:.4f})")

## 4. Train confidence regressor (placeholder target)

The confidence estimator's true target is task accuracy on labeled QA pairs — we don't have those yet. As a placeholder we fit a `GradientBoostingRegressor` against a hand-crafted target (dominant epistemic mass blended with mean evidence score, minus a contradiction penalty). Re-train this once you have real labeled QA pairs.

In [ ]:
import joblib
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

from pipeline.orchestrator import Pipeline as ReasonerPipeline
from pipeline.features import confidence_features

reasoner = ReasonerPipeline()
print("loaded model status:", reasoner.status())

X_feat, y_target = [], []
for r in rows:
    resp = reasoner.run(r["text"])
    feat = confidence_features(resp.epistemic, resp.evidence)
    g = float(r["g"]); y = float(r["y"]); rd = float(r["r"])
    dominance = max(g, y, rd)
    scores = [rec.score for rec in resp.evidence.records] or [0.0]
    avg_score = float(np.mean(scores))
    penalty = 0.1 * len(resp.evidence.contradictions)
    target = max(0.0, min(1.0, 0.5 * dominance + 0.5 * avg_score - penalty))
    X_feat.append(feat)
    y_target.append(target)

Xc = np.stack(X_feat)
yc = np.array(y_target)
print(f"confidence training: X {Xc.shape}, y {yc.shape}")

reg = GradientBoostingRegressor(max_depth=3, n_estimators=200, random_state=0)
reg.fit(Xc, yc)
print(f"train R^2: {reg.score(Xc, yc):.4f}")

joblib.dump(reg, PROJECT / "models/confidence.pkl")
print("saved confidence.pkl")

## 5. Package and download

Writes `manifest.json`, zips `models/`, triggers a download (on Colab).

After the file downloads, on your laptop:
```bash
cd "~/Documents/GitHub/no transformer "
unzip ~/Downloads/models.zip -d models/
```

Then in your CLI, type `:reload`. `:status` should show `[ok]` for every component.

In [ ]:
import json
import time
import zipfile

models_dir = PROJECT / "models"
manifest = {
    "trained_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "files": [],
}
for f in sorted(models_dir.iterdir()):
    if f.is_file() and f.name != "manifest.json":
        manifest["files"].append({"name": f.name, "size_bytes": f.stat().st_size})

(models_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

zip_path = PROJECT / "models.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in models_dir.iterdir():
        if f.is_file():
            z.write(f, f.name)

print(f"built {zip_path}")
print(json.dumps(manifest, indent=2))

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))